In [ ]:
from geofeatureviz.data import OverpassAPIHandler, anki_connector, path_settings

# Rivers data creation
This notebook explains how I get all the River names from my Anki deck for which I want geographical information from OpenStreetMap.

## River Names
The geographical information will be gathered from OSM using the Overpass API; and for some rivers, the names in OSM are in a different language, so they have to be changed so that they can be found in the API-request.

In [ ]:
deck_name = "Allgemeinwissen::01 🟦 Geografie 🌍::01.04 Flüsse Deutschlands 🏞️🇩🇪"
anki_rivers_df = anki_connector.deck_to_df(deck_name)
anki_rivers_df = anki_rivers_df[anki_rivers_df["NoteType"] == "Adrian::Flüsse"]

# Some rivers have different names at OSM (mainly language reasons)
anki_rivers_df["osm_name"] = anki_rivers_df["Flussname"].replace(
    {
        "Donau": "Danube",
        "Eger": "Ohře",
        "Mosel": "La Moselle",
        "Oder": "Odra",
        "Saar": "La Sarre",
    }
)
anki_rivers_df = anki_rivers_df.sort_values(by="osm_name")
anki_rivers_df = anki_rivers_df.reset_index(drop=True)
osm_name_to_name = dict(zip(anki_rivers_df["osm_name"], anki_rivers_df["Flussname"]))
anki_rivers_df

I will use this query to gather river information from OSM using the Overpass API. Since the query is rather long, it might fail; but after some retries, it always works for me. I saved the obtained JSON-file (which is not a GeoJSON!) to avoid requesting the API.

Using the names to get the rivers can be ambiguous (e.g., different rivers having the same name). OSM handles this by giving unique IDs to all objects. With this query, our goal is to find out the IDs of all rivers that we want in our dataset.

**Side note:** the online tool [Overpass Turbo](https://overpass-turbo.eu/) allows easy access to the Overpass API. You can paste the query there and then obtain a GeoJSON-file via *Export -> GeoJSON -> Download*.

In [ ]:
# create query
regex = [f"^{f}$" for f in anki_rivers_df["osm_name"]]
regex = "|".join(regex)
overpass_query_names = f'relation["waterway"="river"]["name"~"{regex}"]'

# run query
api_handler_river_name = OverpassAPIHandler(
    file_path=path_settings.data_raw_dir
    / "osm_rivers"
    / "german_rivers_body_name.json",
)
api_handler_river_name.create_query(overpass_query_names)
river_name_json = api_handler_river_name.get()

river_name_df = api_handler_river_name.parse_json(river_name_json).sort_values(
    by="name"
)

river_name_df

## River IDs

First, we take a look at the river names that appear more than once in our API-response. From a manual inspection of them on openstreetmap.org, I identified the ones that I don't want. I remove them from the list and check that all remaining rivers are the ones we also have in the Anki-Deck.

In [ ]:
# show duplicates
river_name_df[river_name_df.duplicated(subset="name", keep=False)]

In [ ]:
unwanted_ids = [15074211, 15347024, 5213209, 6799134, 7298530]
river_id_df = river_name_df[~river_name_df["id"].isin(unwanted_ids)]
river_id_df = river_id_df.reset_index(drop=True)
assert all(river_id_df.name == anki_rivers_df.osm_name)

river_id_df

I format the DataFrame and save it as CSV-file, so that the names and OSM-IDs can be accessed in a script that loads the geometries from the Overpass API.

In [ ]:
river_df = river_id_df[["id", "name"]]
river_df = river_df.rename(columns={"id": "osm_id"})
river_df["osm_name"] = river_df["name"]
river_df["name"] = river_df["osm_name"].replace(osm_name_to_name)
river_df["osm_type"] = "relation"

# reorder columns
river_df = river_df[["name", "osm_id", "osm_name", "osm_type"]]
river_df = river_df.sort_values(by="name").reset_index(drop=True)

river_df.to_csv(str(path_settings.german_rivers_osm_id_path))